# 🤖 Simple RAG Demo — HR Policy Assistant

## 📚 What is RAG?

RAG stands for **Retrieval-Augmented Generation**.

In plain English:

- We take a document, here an HR policy handbook.
- We chop it into small pieces and turn each piece into a list of numbers called an **embedding** that captures its meaning.
- We store those pieces in a searchable database called a **vector store**.
- When a user asks a question, we search the database for the most relevant pieces and hand them to an AI model to generate a final answer.

> That's it. No magic — just **search + a language model**.

## 🛠️ What We Use

### 📄 Data

`data/hr_policy.txt`

A sample HR policy document.

### 🔢 Embeddings

**Jina AI**

Used to convert text chunks into numerical vectors that capture semantic meaning.

### 🗄️ Vector Store

**FAISS**

Used to store and search the generated embeddings for relevant document chunks.

### 🧠 LLM

**Groq**

Used to generate the final answer based on the retrieved HR policy content.

### 🕸️ Framework

**LangChain**

Used to connect the document processing, embeddings, retrieval, and LLM components.

## ⚙️ Before Running

Make sure:

1. The notebook kernel is set to the project virtual environment.
2. Your `.env` file exists in the project folder.
3. The `.env` file contains:

```text
GROQ_API_KEY=your_groq_api_key
JINA_API_KEY=your_jina_api_key

## Step 1 — Import everything we need

In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")

### LOADING DATA

In [4]:
file_path = os.path.join("data", "hr_policy.txt")
print(file_path)

data\hr_policy.txt


In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

C:\Users\hp\AppData\Local\Temp\ipykernel_2032\1979811456.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [7]:
print(documents)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probati

In [8]:
len(documents)

1

In [11]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [12]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [13]:
print(f"Total Characters in documents: {len(documents[0].page_content)}")

Total Characters in documents: 2597


### SPLITTING DATA 

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, 
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [16]:
print(len(chunks))

9


In [21]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [25]:
print(chunks[6].page_content)

6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.


In [26]:
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)


--- Chunk 1 ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

--- Chunk 2 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Chunk 3 ---
2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

--- Chunk 4 ---
3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are no

### EMBEDDING

In [7]:
## using JINA EMBEDDINGS 

from langchain_community.embeddings import JinaEmbeddings

embeddings = JinaEmbeddings(
    model_name= "jina-embeddings-v5-text-small"
)

print("EMB MODEL NAME: ",embeddings.model_name)

EMB MODEL NAME:  jina-embeddings-v5-text-small


In [29]:
embedding = embeddings.embed_query(chunks[0].page_content)

print(embedding)

[-0.01980623, 0.00328151, -0.03750292, -0.05250409, 0.0867255, -0.12750992, -0.05859831, -0.01546995, 0.07266191, 0.02179857, -0.00098885, 0.00050175, -0.00585983, -0.02894757, -0.01570435, -0.02285334, -0.03281505, -0.02941635, -0.0311743, -0.01500117, -0.03679974, -0.07500584, -0.02847878, -0.05344166, -0.00750058, 0.02836158, 0.03750292, 0.00166273, 0.10266424, 0.04758183, 0.06984919, -0.02894757, 0.03000234, 0.02379091, 0.01652472, -0.02578326, -0.014005, -0.01365341, -0.0209782, 0.04758183, -0.01265724, 0.02965075, -0.07781856, -0.04547229, -0.00024355, 0.05414484, -0.05391045, 0.02379091, 0.04430033, 0.01875146, -0.02894757, -0.01711071, -0.06844283, 0.02543167, 0.04195639, -0.01066489, -0.03422141, 0.02894757, 0.0278928, 0.02812719, 0.02132978, 0.02824439, 0.02062661, 0.01570435, 0.00855535, 0.03375263, -0.01922025, 0.02672083, -0.02906476, 0.03679974, -0.0104891, 0.03258066, -0.03562777, -0.02121259, 0.00890694, -0.04992576, 0.00571334, -0.06656768, 0.01488397, -0.04008124, -0.

### STORE DATA IN VECTOR DB

In [8]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents = chunks, embedding = embeddings )

print("CHUNKS ARE STORED: ", vector_store.index.ntotal)

CHUNKS ARE STORED:  9


In [ ]:
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query ,k=2)
print(f"Query: {test_query}\n")
for i, match in enumerate(top_matches, start = 1):
    print(f"--- match {i}---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- match 1---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- match 2---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [39]:
query = "Can employees work from home?"

matches = vector_store.similarity_search(query, k = 1)
print(f"QUERY: {query}\n")

for i, match in enumerate (matches,start=1):
    print(f"---match--- {i}")
    print(match.page_content)
    print()


QUERY: Can employees work from home?

---match--- 1
2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.



### TOOL

In [15]:
retriever = vector_store.as_retriever(search_kwargs={"k":2})

In [ ]:
def search_hr_policy(questions:str) -> str:
    """Search the HR Policy document for information about leave, work from home,
    probation notice period, reimbursement, code of conduct, holidays, or exit process."""
    matching_chunks = retriever.invoke(questions)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

#### DATA RETRIEVAL 

### LLM

In [ ]:
from langchain_groq import ChatGroq 

llm = ChatGroq( 
    model = "openai/gpt-oss-20b",
    temperature =  0
)

llm.model_name 

'openai/gpt-oss-20b'

In [10]:
response = llm.invoke("Hey learning rag is hard tell me in one line")
print(response.content)

Learning RAG is hard because it requires mastering both sophisticated retrieval mechanisms and seamless integration with generative models, each with its own complex tuning, data‑quality, and alignment challenges.


### AI AGENT

In [11]:
from langchain.agents import create_agent

In [18]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """


    You are a friendly HR Assistant.
    always use the search_hr_policy tool to look up
    facts before answering.
    If the answer isn't in the search result, say you don't know "
    insted of guessing"
    """
)



In [19]:
def as_hr_assistant(questions: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("Question", questions)

    response = hr_assistant.invoke({"messages": [{"role" : "user", "content": questions}]})
    answer = response["messages"][-1].content

    print("Answer", answer)
    print()
    return answer

In [29]:
responses = hr_assistant.invoke(
    {
        "messages" : [
            {
                "role" : "user",
                "content":"Tell me about leave policy how to applied for leave.?"
            }
        ]
    }
)

In [36]:
print(responses)

{'messages': [HumanMessage(content='Tell me about leave policy how to applied for leave.?', additional_kwargs={}, response_metadata={}, id='e5de9d8a-44d0-4b14-ad0a-b2fb9d0a3c67'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use search_hr_policy tool. Provide query about leave policy.', 'tool_calls': [{'id': 'fc_015c04f3-9631-4dc4-9ae2-ca808c6e3881', 'function': {'arguments': '{"questions":"leave policy how to applied for leave"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 206, 'total_tokens': 252, 'completion_time': 0.049083193, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.022981623, 'prompt_tokens_details': None, 'queue_time': 0.311857023, 'total_time': 0.072064816}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4a35f7bd1b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'},

In [ ]:
print(responses["messages"][-1].content)

Sure thing! Here’s what our leave policy says and how you can apply:

| **Leave Type** | **Entitlement** | **How to Apply** |
|----------------|-----------------|------------------|
| **Annual (Paid) Leave** | 20 days per calendar year | Submit a request through the HR portal **at least 5 working days in advance**. |
| **Carry‑over** | Up to 5 unused days can be carried forward to the next year | No extra action needed—just make sure you’ve used the carry‑over days before the new year starts. |
| **Sick Leave** | 10 paid days per year | For sick leave longer than 2 consecutive days, you’ll need to upload a medical certificate in the HR portal. |

**Quick Steps to Apply for Leave:**

1. **Log in** to the HR portal.  
2. Navigate to the **Leave Request** section.  
3. Select the type of leave (Annual or Sick).  
4. Pick the start and end dates.  
5. If it’s sick leave > 2 days, attach the medical certificate.  
6. Click **Submit**. Your manager will receive a notification and can approve